# Word Embeddings
This notebook has a function called "get_embeddings" in the end which takes an input file and generates the embeddings for each sentence and saves it in a file

### 0. Imports 

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/drive/


In [2]:
% cd /content/drive/My Drive/Thesis_colab

/content/drive/My Drive/Thesis_colab


In [3]:
! pip install transformers
! pip install nltk
! pip install sentence_transformers
! pip install stop_words
! pip install langdetect

     |████████████████████████████████| 890kB 3.0MB/s 
     |████████████████████████████████| 1.1MB 17.1MB/s 
     |████████████████████████████████| 3.0MB 19.5MB/s 
     |████████████████████████████████| 890kB 30.1MB/s 
  Created wheel for sacremoses: filename=sacremoses-0.0.43-cp36-none-any.whl size=893257 sha256=0f9a334256324de57081b3c92f6ef93de3d071807bcee4dfe98a596d3efd74ee
  Stored in directory: /root/.cache/pip/wheels/29/3c/fd/7ce5c3f0666dab31a50123635e6fb5e19ceb42ce38d4e58f45
Successfully built sacremoses
     |████████████████████████████████| 71kB 2.2MB/s 
     |████████████████████████████████| 778kB 7.6MB/s 
     |████████████████████████████████| 3.0MB 29.9MB/s 
  Created wheel for sentence-transformers: filename=sentence_transformers-0.3.5.1-cp36-none-any.whl size=100387 sha256=d416bfd21ba7bfd44f73c0ff3bb61d29a6a51d0ea37249b7739c0a0614537228
  Stored in directory: /root/.cache/pip/wheels/1a/05/25/5201c408b8048c9663d75c82dc079cb72633cf543af1aa84da
Successfully built sent

In [54]:
# importing all necessary modules 
import pandas as pd
from nltk import sent_tokenize
import torch
from transformers import XLNetTokenizer, XLNetModel, BertTokenizer, BertModel, XLMTokenizer, XLMModel, ElectraTokenizer, ElectraModel, AlbertTokenizer, AlbertForMaskedLM
import numpy as np
import os
from utilities.document_preprocessing import preprocess_documents
from sentence_transformers import SentenceTransformer, util
from datetime import datetime
import json

In [5]:
#needs to be run only once in the beginning
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

### Function Definations

In [57]:
def preprocess(text_column, model_name=None):
    '''
    Takes in a pandas column containing text and preprocesses it.
    Also creates segment ids if it is required by the model

    - Inputs:
        text_column (pandas series): The column containing all text entries to be processed
        model_name (str): name of the model to be used for generating embeddings

    - Output:
        preprocessed_indexes_column (list): a list of all text tokenized, encoded and padded.

    '''

    # Load pre-trained model tokenizer (vocabulary)
    if model_name == 'bert':
        tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')
        pad_id = 0
    elif model_name == 'xlnet':
        tokenizer = XLNetTokenizer.from_pretrained('xlnet-large-cased')
        pad_id = tokenizer.pad_token_id
    elif model_name == 'xlm':
        tokenizer = XLMTokenizer.from_pretrained('xlm-mlm-en-2048')
        pad_id = tokenizer.pad_token_id
    elif model_name == 'electra':
        tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
        pad_id = tokenizer.pad_token_id
    elif model_name == 'albert':
        tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')
        pad_id = tokenizer.pad_token_id

    else:
        print('Please Specify the appropriate model name.')

    # convert the pandas series to a list for the sake of easier processing
    text_column = list(text_column)

    # define place holders
    preprocessed_indexes_column = []

    
    max_length = 0
    for text in text_column: # iterate through each objective
        print (len(text.split()))
        if len(text.split()) > 500:

          text = text[:500] # if the objective has more than 500 words, only take the first 500 words
          print (len(text.split()))
        # tokenize the sentence into words using the model specific tokenizer
        tokenized_text = tokenizer.tokenize(text)

        # Map the token strings to their vocabulary indices.
        indexed_tokens = tokenizer.encode(tokenized_text, add_special_tokens=False)
        preprocessed_indexes_column.append(indexed_tokens)

        # keeping track of the maximum sequence length
        max_length = len(tokenized_text) if len(tokenized_text) > max_length else max_length


    # pad the sequence to allow batch processing
    preprocessed_indexes_column = pad_sequences(preprocessed_indexes_column, max_length, pad_id)
    return preprocessed_indexes_column


In [42]:
def pad_sequences(data, max_length, pad_id):
    '''
    Pads all the  sequence in 'data' with 'pad_id' to have a constant length of 'max_length'
    - Inputs:
        text_column (pandas series): The column containing all text entries to be processed
        model_name (str): name of the model to be used for generating embeddings

    - Output:
        preprocessed_indexes_column (list): a list of all text tokenized, encoded and padded.

    '''
    data_paded = []
    for d in data:
        if len(d) < max_length:
            temp = d + [pad_id] * (max_length - len(d))
            data_paded.append(temp)
        else:
            data_paded.append(d)
    return data_paded

In [ ]:
def generate_embedding(objectives, model_name, batch_size=100):
    '''
    Takes in a pandas dataframe and generates embeddings for the text column
    - Inputs:
        pd_dataframe (pandas dataframe): The dataframe containing all text column and their ids
        model_name (str): name of the model to be used for generating embeddings
        batch_size (int): batch size to use when generating embeddings for sentences

    - Output:
        sentence_embedding (tensor): tensor of shape n by 1024 where n is the number of sentence

    '''

    if model_name == 'bert':
        # Load pre-trained bert model (weights)
        model = BertModel.from_pretrained('bert-large-uncased')
    elif model_name == 'xlnet':
        # Load pre-trained xlnet model (weights)
        model = XLNetModel.from_pretrained('xlnet-large-cased')
    elif model_name == 'xlm':
        # Load pre-trained xlm model (weights)
        model = XLMModel.from_pretrained('xlm-mlm-en-2048')
    elif model_name == 'electra':
        # Load pre-trained electra model (weights)
        model = ElectraModel.from_pretrained('google/electra-small-discriminator')
    elif model_name == 'albert':
        # Load pre-trained albert model (weights)
        model = AlbertForMaskedLM.from_pretrained('albert-base-v2')
    


    sentences_per_batch = batch_size

    

    #setting up the device
    if torch.cuda.is_available():
        dev = "cuda:0"
    else:
        dev = "cpu"
    device = torch.device(dev)
    print (device)

    # Put the model in "evaluation" mode, meaning feed-forward operation.
    model.eval()
    model.to(device)
    num_sentences = len(objectives)
    sentence_embedding = []
    
    if num_sentences > sentences_per_batch:
        num_batches = num_sentences//sentences_per_batch

        for i in range(num_batches):
            start = i * sentences_per_batch
            end = (i+1)* sentences_per_batch
            if i == num_batches-1:
                end = num_sentences
            mini_objective = list(objectives[start:end])
            
            # Convert inputs to PyTorch tensors
            tokens_tensor = torch.tensor([mini_objective]).squeeze()
            tokens_tensor = tokens_tensor.to(device)

            # Predict hidden states features for each layer
            with torch.no_grad():
                encoded_layers = model(tokens_tensor)

            # taking embeddings of the last layer.
            # token_vecs` is a tensor with shape [n x k x 1024]
            token_vecs = encoded_layers[0]

            # Calculate the average of all k  token vectors and adding to the main list
            sentence_embedding += torch.mean(token_vecs, dim=1).tolist()
            print ("Embedding for batch {} out of {} batches Completed.".format(i, num_batches))
    else:
        # Convert inputs to PyTorch tensors
        print (np.array(objectives)[0])
        tokens_tensor = torch.tensor([objectives]).squeeze()
        tokens_tensor = tokens_tensor.to(device)

        # Predict hidden states features for each layer
        with torch.no_grad():
            encoded_layers = model(tokens_tensor)

        # taking embeddings of the last layer.
        # token_vecs` is a tensor with shape [n x k x 1024]
        token_vecs = encoded_layers[0]
        sentence_embedding = torch.mean(token_vecs, dim=1).tolist()
    print("Our final sentence embedding vector of shape:", len(sentence_embedding))
    return (sentence_embedding)


In [25]:

def generate_sentence_embeddings(objectives, model_name, batch_size=100):
    '''
    Takes in a pandas dataframe and generates embeddings for the text column using sentence_transformer
    - Inputs:
        pd_dataframe (pandas dataframe): The dataframe containing all text column and their ids
        model_name (str): name of the model to be used for generating embeddings
        batch_size (int): batch size to use when generating embeddings for sentences

    - Output:
        sentence_embedding (tensor): tensor of shape n by 1024 where n is the number of sentence

    '''

    model2transformer = {'bert': 'sentence-transformers/bert-base-nli-mean-tokens', 
                         'roberta':'sentence-transformers/roberta-large-nli-stsb-mean-tokens',
                         #'roberta-base':'roberta-base-nli-stsb-mean-tokens',
                         'distilbert':'distilbert-base-nli-stsb-mean-tokens'}


    # get the specfic model name for sentence transformer
    model_name = model2transformer[model_name]
    embedder = SentenceTransformer(model_name)
    sentences_per_batch = batch_size

    # setting up the device
    if torch.cuda.is_available():
        dev = "cuda:0"
    else:
        dev = "cpu"
    device = torch.device(dev)
    print(device)

    num_sentences = len(objectives)
    sentence_embedding = []

    if num_sentences > sentences_per_batch:
        num_batches = num_sentences // sentences_per_batch

        for i in range(num_batches):
            start = i * sentences_per_batch
            end = (i + 1) * sentences_per_batch
            if i == num_batches - 1:
                end = num_sentences
            mini_objective = list(objectives[start:end])

            encoded_layers = embedder.encode(mini_objective, convert_to_tensor=True)
            


            # Calculate the average of all k  token vectors and adding to the main list
            sentence_embedding += encoded_layers.cpu().tolist()
            print("Embedding for batch {} out of {} batches Completed.".format(i, num_batches))
    else:


        encoded_layers  = embedder.encode(objectives, convert_to_tensor=True)

        sentence_embedding = encoded_layers.cpu().tolist()


    print("Our final sentence embedding vector of shape:", len(sentence_embedding))
    return (sentence_embedding)


In [55]:
def get_embeddings(model_name = None, file_name =None, sentences= None, saved_embedding_dir = None, batch_size = 100 , dr = None):
    '''
    wrapper functions which generates the the embeddings for the text in input file and saves in a json file
    - Inputs:
        file_name (str): string pointing to the file to be read
        model_name (str): name of the model to be used for generating embeddings
        saved_embedding_dir (str): path to json file, if the precomputed embedding needs to be used.
        sentences (list of list): sentences, if data has been read previously
        bath_size (int): batch size to use when generating the embeddings
        dr (str): folder path where the embedding should be stored. if not given, the default path is used

    - Output:
        sentence_embedding (tensor): tensor of shape n by 1024 where n is the number of sentence

    '''

    if saved_embedding_dir is not None:
        with open(saved_embedding_dir) as json_file:
            sentence_embedding = json.load(json_file)
            return sentence_embedding

    if file_name is None and sentences is None:
        print ('Either a file name or a list of sentences must be provided')
        return

    pd_objectives = pd.DataFrame()
    # Read file if file path is provided, read the data from there and preprocess it using the pre-defined utility function
    if file_name is not None:
        _, sentences, _ = preprocess_documents(file=file_name, verbose=False)

    pd_objectives['objectives'] = sentences

    
    # generate embeddings
    
    sentence_transformers_model = ['bert', 'roberta', 'roberta-base', 'distilbert']
    hugging_face_models = ['xlnet', 'electra', 'xlm', 'albert']

    if model_name in sentence_transformers_model: #use sentence transformer library for it
      sentence_embedding = generate_sentence_embeddings(pd_objectives['objectives'], model_name, batch_size)
    
    else: #for xlnet, use the hugging face library
      # do model specific preprocessing for embedding generation
      pd_objectives['objectives'] = preprocess(pd_objectives['objectives'], model_name)

      sentence_embedding = generate_embedding(pd_objectives['objectives'], model_name, batch_size)

    # save embeddings

    if dr == None:
        output_folder = 'output/embeddings/' +  str(datetime.now().strftime("%Y_%m_%d_%H_%M_%S") ) + '_'  + model_name + '/'
    else:
        output_folder = dr
    file_name = output_folder + model_name + '_embeddings.json'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    with open(file_name, 'w') as fp:
        json.dump(sentence_embedding, fp)

    return sentence_embedding


possible models: 'bert', 'xlnet', 'roberta', 'distilbert', 'xlm', 'electra', 'albert'

In [52]:
file = "data/PMP_Skills_Matching_Pilot_2018_2019.xlsx"

In [53]:
xlm_sentence_embedding = get_embeddings(file_name= file, model_name='xlm', batch_size = 100)

Reading the file
Preprocessing raw texts 

Couldn't detect the language of the text on index 2264
 
  
 
Couldn't detect the language of the text on index 2265
 
  
 
Couldn't detect the language of the text on index 2266
 
  
 
Couldn't detect the language of the text on index 2267
 
  
 
Couldn't detect the language of the text on index 2268
 
  
 
Couldn't detect the language of the text on index 2269
 
  
 
Couldn't detect the language of the text on index 2271
 
  
 
Couldn't detect the language of the text on index 2272
 
  
 
Couldn't detect the language of the text on index 2273
 
  
 
Couldn't detect the language of the text on index 2275
 
  
 
Couldn't detect the language of the text on index 2276
 
  
 
Couldn't detect the language of the text on index 2277
 
  
 
Couldn't detect the language of the text on index 2278
 
  
 
Couldn't detect the language of the text on index 2279
 
  
 
Couldn't detect the language of the text on index 2280
 
  
 
Couldn't detect the languag

In [ ]:
electra_sentence_embedding = get_embeddings(file_name= file, model_name='electra', batch_size = 500)

Reading the file
Preprocessing raw texts 

Couldn't detect the language of the text on index 2264
 
  
 
Couldn't detect the language of the text on index 2265
 
  
 
Couldn't detect the language of the text on index 2266
 
  
 
Couldn't detect the language of the text on index 2267
 
  
 
Couldn't detect the language of the text on index 2268
 
  
 
Couldn't detect the language of the text on index 2269
 
  
 
Couldn't detect the language of the text on index 2271
 
  
 
Couldn't detect the language of the text on index 2272
 
  
 
Couldn't detect the language of the text on index 2273
 
  
 
Couldn't detect the language of the text on index 2275
 
  
 
Couldn't detect the language of the text on index 2276
 
  
 
Couldn't detect the language of the text on index 2277
 
  
 
Couldn't detect the language of the text on index 2278
 
  
 
Couldn't detect the language of the text on index 2279
 
  
 
Couldn't detect the language of the text on index 2280
 
  
 
Couldn't detect the languag

In [ ]:
albert_sentence_embedding = get_embeddings(file_name= file, model_name='albert', batch_size = 500)

2048